In [18]:
import re
import glob
import os
import pandas as pd


BOX_RE = re.compile(r'^\s*(?:MECIR\s+Box|Box)\b', re.I)

In [19]:
def merge_boxes_in_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Expects df with columns: ['section', 'paragraph'].
    Merges a 'Box ...' row with consecutive bullet rows (starting with '•') in the same section.
    """
    rows = []
    i = 0
    n = len(df)

    while i < n:
        section = str(df.at[i, 'section'])
        para = str(df.at[i, 'paragraph'])

        if BOX_RE.match(para or ""):
            # Start aggregating this box
            collected = [para.strip()]
            i += 1
            # Consume consecutive bullet rows in the same section
            while i < n:
                s2 = str(df.at[i, 'section'])
                p2 = str(df.at[i, 'paragraph'] or "")
                # stop if section changes or we hit another "Box ..." (new box)
                if s2 != section or BOX_RE.match(p2):
                    break
                # keep only bullets (• ...)
                if p2.strip().startswith('•'):
                    collected.append(p2.strip())
                    i += 1
                    continue
                # any non-bullet paragraph ends the box
                break

            rows.append({'section': section, 'paragraph': "\n".join(collected)})
        else:
            rows.append({'section': section, 'paragraph': para})
            i += 1

    out = pd.DataFrame(rows, columns=['section', 'paragraph']).drop_duplicates().reset_index(drop=True)
    return out

In [20]:
def postprocess(data):
    data = pd.read_csv(data)
    print(f"[INFO] Initial rows: {len(data)}")
    data = merge_boxes_in_df(data)
    print(f"[INFO] Rows after merging boxes: {len(data)}")
    return data

In [21]:
INPUT_DIR = "data/parsed_paragraphs"
OUTPUT_DIR = "data/postprocessed_paragraphs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

files = sorted(set(
    glob.glob(os.path.join(INPUT_DIR, "*.csv"))
))

if not files:
    print("No CSV files found in:", INPUT_DIR)

for path in files:
    base = os.path.splitext(os.path.basename(path))[0]
    out_csv = os.path.join(OUTPUT_DIR, f"{base}_paragraphs.csv")
    try:
        df = postprocess(path)
        df.to_csv(out_csv, index=False, encoding="utf-8")
        print(f"[OK] {base}: {len(df)} rows -> {out_csv}")
    except Exception as e:
        print(f"[ERROR] {base}: {e}")

[INFO] Initial rows: 352
[INFO] Rows after merging boxes: 345
[OK] Chapter 10_ Analysing data and undertaking meta-analyses _ Cochrane_paragraphs: 345 rows -> data/postprocessed_paragraphs\Chapter 10_ Analysing data and undertaking meta-analyses _ Cochrane_paragraphs_paragraphs.csv
[INFO] Initial rows: 287
[INFO] Rows after merging boxes: 287
[OK] Chapter 11_ Undertaking network meta-analyses _ Cochrane_paragraphs: 287 rows -> data/postprocessed_paragraphs\Chapter 11_ Undertaking network meta-analyses _ Cochrane_paragraphs_paragraphs.csv
[INFO] Initial rows: 191
[INFO] Rows after merging boxes: 191
[OK] Chapter 12_ Synthesizing and presenting findings using other methods _ Cochrane_paragraphs: 191 rows -> data/postprocessed_paragraphs\Chapter 12_ Synthesizing and presenting findings using other methods _ Cochrane_paragraphs_paragraphs.csv
[INFO] Initial rows: 191
[INFO] Rows after merging boxes: 191
[OK] Chapter 13_ Assessing risk of bias due to missing evidence in a meta-analysis _ Co